# 05. MHD Stability and Perturbed Equilibria

Session 04 measured perturbations: a frequency, a toroidal mode number modulo its alias step,
rational surfaces the mode could sit on. This session asks why such a perturbation can exist at all.

```text
Session 03   equilibrium + kinetic plasma state
      ↓
Session 04   measured perturbations and transient dynamics
      ↓
Session 05   stability + resistive layers + driven 3-D plasma response      <- here
      ↓
Session 06   many discharges: operational space and statistics
```

One scientific chain runs through the session, and every link ends in the same IMAS data model:

```text
EFIT g-file ──CHEASE──▶ refined fixed-boundary equilibrium ──┬──DCON──▶ ideal δW, eigenfunction   ┐
 (free boundary,         (straight-field-line ready)         ├──RDCON─▶ rational surfaces, Δ'     ├─▶ IMAS ─▶ vaft plots
  session 03)                                                └──DCON──▶ GPEC ─▶ 3-D response      ┘   (equilibrium,
                                                                                                       mhd_linear, ntms)
```

One linear operator is asked three questions along it:

$$
\text{equilibrium} \rightarrow \text{linearization} \rightarrow \text{Fourier decomposition}
\rightarrow \text{energy principle}
\;\rightarrow\;
\begin{cases}
\delta W & \text{DCON: ideal stability} \\
q = m/n,\ \eta \neq 0 & \text{RDCON: tearing, } \Delta' \\
\mathbf f_{\mathrm{ext}} \neq 0 & \text{GPEC: driven 3-D perturbed equilibrium}
\end{cases}
$$

## Session Overview

**The question.** Given an equilibrium and a measured or applied perturbation, is the plasma
intrinsically unstable, resistively unstable at a rational surface, or simply responding to an
external 3-D drive?

By the end of this session you will be able to:

- linearize ideal MHD about an axisymmetric equilibrium and say why the displacement
  $\boldsymbol\xi$ is the natural perturbation variable;
- explain why toroidal mode numbers $n$ separate in an axisymmetric equilibrium while the poloidal
  harmonics $m$ of one $n$ couple;
- read the energy principle: a displacement with $\delta W < 0$ means ideal instability;
- say what CHEASE changes about an EFIT equilibrium, and why a straight-field-line stability code
  wants a closed, fixed-boundary one;
- explain why ideal MHD cannot reconnect field lines, what a resistive layer is, and what
  $\Delta'$ measures;
- tell a homogeneous stability problem ($\mathcal L\boldsymbol\xi = 0$, DCON) from a driven one
  ($\mathcal L\boldsymbol\xi = \mathbf f_{\mathrm{ext}}$, GPEC) — the same operator, two questions;
- describe VEST's three 3-D coil sets, build a sector-current pattern for a chosen $n$, and say which
  toroidal harmonics six sectors can and cannot make;
- use linearity to turn two runs into a whole relative-phase scan, and separate the model's
  excitation space from what the VEST power supplies can actually do.

**The equilibrium.** Discharge 39915, the discharge of sessions 02–04, through the same EFIT
reconstruction sessions 03 and 04 read from the reference sample ODS, at 319 ms. DCON cannot trace
that reconstruction's flux surfaces directly (#1246); CHEASE's refinement of it (Step 2) is what every
stability code below reads, and it traces cleanly. Step 1 lays it beside the legacy 2023 g-file of
the same instant: the rational surfaces a stability code sees are only as good as the reconstruction
they come from.

**Two modes, one data path.** CHEASE, DCON, RDCON and GPEC are external Fortran codes. *Offline*
(the default) every solver stage loads a **precomputed, validated IMAS sample** of exactly the run a
lab session makes — `vaft/data/samples/39915/mhd_stability/`, built by
`workflow/tutorial_05_mhd_samples/build_mhd_stability_sample.py`. *Lab*
(`VAFT_TUTORIAL_MODE=lab`) runs a stage when its executables are found and launchable, maps the
result to the same IMAS IDS, and falls back to the sample — saying so — when it cannot. Every plot
downstream reads the IMAS data, never the solver files, so the two modes draw the same figures.
The lab path takes about ten minutes.

## Physical Context

### 1. From a measured perturbation to three possibilities

Session 04 ended with observables: $f$, $n$ (modulo the alias step), a $\delta B$ amplitude, SXR and
camera structure, a timing. They admit three broad explanations:

| explanation | what drives it | the question a code answers |
| --- | --- | --- |
| intrinsic **ideal** instability | pressure and current gradients, no reconnection | can a displacement lower the ideal-MHD energy? |
| **resistive tearing** instability | the same free energy, released by reconnection at $q = m/n$ | is the tearing index $\Delta'$ positive? |
| **driven 3-D response** | an external non-axisymmetric field (coils, error fields) | how does a stable plasma respond to a known drive? |

### 2. Flux coordinates, straight field lines and rational surfaces

An axisymmetric equilibrium nests its field lines on flux surfaces labelled by $\psi$. Choose a
poloidal angle $\theta$ in which every field line is *straight*,

$$
\frac{d\phi}{d\theta}\Big|_{\text{field line}} = q(\psi) \quad\text{on each surface},
$$

and a helical perturbation $e^{i(m\theta - n\phi)}$ has the field-line pitch exactly where
$q(\psi_s) = m/n$ — the **rational surface**. Straight-field-line (SFL) angles — PEST, Hamada,
Boozer — exist only on closed, nested surfaces with a finite $q$: they break down on the
separatrix, where $q \to \infty$, and they inherit any roughness of the equilibrium's $\psi$. That
is why the stability codes below are fed a *refined, fixed-boundary* equilibrium rather than the
reconstruction itself (Step 2).

### 3. Linearization about an equilibrium

Session 03 wrote force balance, $\mathbf J_0\times\mathbf B_0 = \nabla p_0$. Perturb it,
$\mathbf B = \mathbf B_0 + \delta\mathbf B$, $\mathbf J = \mathbf J_0 + \delta\mathbf J$,
$p = p_0 + \delta p$, and describe the motion by the plasma **displacement** $\boldsymbol\xi(\mathbf x, t)$.
In ideal MHD the field is frozen into the moving fluid, so

$$
\delta\mathbf B = \nabla\times(\boldsymbol\xi\times\mathbf B_0),
\qquad
\rho_0\,\frac{\partial^2\boldsymbol\xi}{\partial t^2} = \mathcal F[\boldsymbol\xi],
$$

where $\mathcal F$ is the linear ideal-MHD force operator. It is built from the equilibrium alone:
the equilibrium decides which perturbations are possible.

### 4. Fourier decomposition

In an axisymmetric equilibrium, $\phi$ does not appear in $\mathcal F$, so each toroidal mode
number $n$ is a separate problem. The poloidal angle does appear (through $B_0(\theta)$, the shape),
so the poloidal harmonics of one $n$ **couple**:

$$
\boldsymbol\xi = \sum_n \Big[\sum_m \boldsymbol\xi_{m,n}(\psi)\, e^{i m\theta}\Big] e^{-i n\phi}\, e^{-i\omega t}.
$$

A tokamak eigenmode is therefore not one $(m, n)$ component. It is one $n$ with a spectrum of $m$ —
the "dominant $m$" DCON reports is the largest term, not the only one.

### 5. An eigenvalue problem

With $\boldsymbol\xi \propto e^{-i\omega t}$, $\;-\rho_0\omega^2\boldsymbol\xi = \mathcal F[\boldsymbol\xi]$.
$\mathcal F$ is self-adjoint, so $\omega^2$ is real:

- $\omega^2 > 0$ — the perturbation oscillates: stable;
- $\omega^2 < 0$ — it grows exponentially: unstable;
- $\omega^2 = 0$ — marginal stability.

### 6. The energy principle

$$
\delta W[\boldsymbol\xi] = -\frac12\int \boldsymbol\xi^*\cdot\mathcal F[\boldsymbol\xi]\,dV .
$$

If **any** admissible displacement has $\delta W < 0$, the equilibrium is ideally unstable; if
$\delta W > 0$ for all of them, it is stable. Force operator, eigenvalue problem and energy principle
are three representations of one linear problem. DCON solves it for each $n$ and reports the
energy eigenvalues: plasma ($W_p$), vacuum ($W_v$) and total ($W_t$).

Local limits of the same principle — the Mercier (interchange) criterion and high-$n$ ballooning —
are evaluated surface by surface rather than globally.

### 7. Why ideal MHD cannot tear

The ideal Ohm's law $\mathbf E + \mathbf v\times\mathbf B = 0$ freezes the field into the fluid:
field-line topology cannot change. At a rational surface, $q(\psi_s) = m/n$, a helical perturbation
with the same pitch as the field lines can only be resisted by bending them. Breaking and
reconnecting them — an island — needs something ideal MHD does not have.

### 8. Resistive layers and $\Delta'$

With resistivity, $\mathbf E + \mathbf v\times\mathbf B = \eta\mathbf J$. Away from the rational
surface $\eta$ is negligible and the ideal ("outer") solution holds; in a thin layer around it,
$\eta$ allows reconnection. The two solutions are **matched**. The outer solution's information is
one number per surface, the jump of its logarithmic derivative:

$$
\Delta' = \left[\frac{1}{\tilde\psi}\frac{d\tilde\psi}{dr}\right]_{r_s^-}^{r_s^+}.
$$

In the classical single-surface picture, $\Delta' > 0$ drives tearing. A toroidal plasma has many
rational surfaces per $n$ and they couple, so the real object is a **matrix** of $\Delta'$, and its
diagonal alone is not the whole story.

### 9. Stability and perturbed force balance share one operator

Linearize force balance directly: $\delta\mathbf F = \delta\mathbf J\times\mathbf B_0 +
\mathbf J_0\times\delta\mathbf B - \nabla\delta p = \mathcal F[\boldsymbol\xi]$. At marginal stability,
$\omega^2 = 0$, the dynamic equation becomes

$$
\boxed{\mathcal F[\boldsymbol\xi] = 0}
$$

— a static, linearly perturbed equilibrium. So the same operator answers two questions:

```text
             same linear ideal-MHD operator
                          |
          +---------------+----------------+
          |                                |
   homogeneous  L xi = 0             driven  L xi = f_ext
   does the equilibrium admit a      how does a stable plasma answer
   self-growing perturbation?        a known external 3-D field?
          |                                |
        DCON                             GPEC
```

A general dynamic eigenmode is not a static perturbed equilibrium; the identity is exact in the
marginal, $\omega \to 0$ limit, which is exactly where a slowly applied coil field lives.

### 10. GPEC: a linear 3-D perturbed equilibrium

GPEC keeps the axisymmetric equilibrium as the background and computes the linear response to a
small applied field, $\mathbf B = \mathbf B_0 + \delta\mathbf B_{3D}$ with $|\delta B| \ll |B_0|$. It
is not a nonlinear 3-D equilibrium solver. Its central decomposition is

$$
\delta\mathbf B_{\mathrm{total}} = \delta\mathbf B_{\mathrm{vacuum}} + \delta\mathbf B_{\mathrm{plasma}} .
$$

The field the coils apply is not the field inside the plasma: the plasma can **screen** a resonant
component (currents on the rational surface cancel it) or **amplify** a component that couples to a
weakly stable kink. Resonant and non-resonant parts both matter; GPEC is not a resonant-field
calculator.

### 11. Which code answers which question

| | CHEASE | DCON | RDCON | GPEC |
| --- | --- | --- | --- | --- |
| model | fixed-boundary Grad–Shafranov | ideal MHD | resistive matching across rational layers | ideal MHD with an external drive |
| equation | $\Delta^*\psi = -\mu_0 R^2 p' - FF'$ | $\mathcal L\boldsymbol\xi = 0$ | outer ideal solution + inner layers | $\mathcal L\boldsymbol\xi = \mathbf f_{\mathrm{ext}}$ |
| key number | GS residual, $q$ profile | $\delta W$ (energy eigenvalues) | $\Delta'$ per surface, and its matrix | vacuum/plasma/total $\delta B_{m,n}$, resonant flux |
| structure | SFL-ready flux surfaces | global eigenfunction $\xi_m(\psi)$ | rational surfaces and their coupling | 3-D field and displacement |
| typical physics | equilibrium consistency | kink, interchange, ballooning | tearing | error-field and RMP response |

RDCON is not a nonlinear time-evolution code: it solves the matching problem. STRIDE solves the same
problem independently and is the natural cross-check of RDCON's $\Delta'$.

## Load / Prepare Data

### Installation and execution

The four solvers are external codes; VAFT prepares their inputs, runs them, reads what they write
and maps it to IMAS. Install them with the VAFT installation guide
([`install/`](https://github.com/VEST-Tokamak/vaft/tree/main/install)) and check the result:

| code | what it does here | install | check |
| --- | --- | --- | --- |
| CHEASE | fixed-boundary Grad–Shafranov refinement of the EFIT equilibrium | `install/install_chease.sh` | `python install/check_chease.py` |
| DCON (+ MATCH) | ideal free-boundary stability, energy eigenvalues, eigenfunction (`solutions.bin`) | `install/install_gpec.sh` | `python install/check_gpec.py` |
| RDCON | resistive matching across rational surfaces, $\Delta'$ | (GPEC suite) | (same) |
| GPEC | linear plasma response to the VEST 3-D coils | (GPEC suite) | (same) |

Windows users run the `.ps1` equivalents. The executables are found through `$CHEASEHOME` and
`$GPECHOME` (`vaft.code.chease.find_chease_executable`, `vaft.code.gpec.find_gpec_executable`).

**One execution policy for the whole notebook**, `VAFT_TUTORIAL_MODE`:

- `offline` (default) — no solver, no network: every stage loads its precomputed IMAS sample;
- `lab` — a stage runs when its executables are found **and launchable** and its upstream stage was
  computed live; otherwise it loads the sample and records why.

Every stage keeps four facts apart, so that a sample standing in for a failed run never reads as a
successful run:

| field | values |
| --- | --- |
| execution | `not attempted`, `computed`, `failed`, `unavailable`, `incomplete` |
| data source | `live`, `precomputed sample` |
| validation | `passed`, `failed`, `not evaluated` |
| fallback reason | why the sample was used, when it was |

In [ ]:
import os
import tempfile
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from omas import ODS

import vaft
from vaft.code import gpec
from vaft.code.chease import CHEASEConfig, find_chease_executable
from vaft.code.chease_scan import EquilibriumVariation, scan_chease
from vaft.code.stage_status import StageStatus, executable_ready, format_stage_table
from vaft.data import read_geqdsk
from vaft.data.eqdsk import from_omas, write_geqdsk
from vaft.data.resources import data_path, mhd_stability_manifest, mhd_stability_sample, sample_geqdsk
from vaft.machine_mapping.mhd_linear import extract_dcon_stability, extract_rdcon_stability
from vaft.omas.vest_upstream import build_gpec_ideal_ods, build_mhd_linear_ods
from vaft.formula import stability
from vaft.machine_mapping.coils_non_axisymmetric import (
    CoilExcitation, apply_coil_excitation, coils_non_axisymmetric,
)
from vaft.machine_mapping.coils_non_axisymmetric_geometry import load_vest_3d_coil_config
from vaft.process.coils_non_axisymmetric import biot_savart_filaments, toroidal_mode_decomposition
from vaft.process.magnetic_island import (
    as_equilibrium, equilibrium_safety_factor, find_rational_surfaces,
)
from vaft.process import perturbation

In [ ]:
MODE = os.environ.get("VAFT_TUTORIAL_MODE", "offline")
if MODE not in ("offline", "lab"):
    raise ValueError(f"VAFT_TUTORIAL_MODE must be 'offline' or 'lab', not {MODE!r}")

STAGES = {name: StageStatus(name) for name in ("CHEASE", "DCON", "RDCON", "GPEC")}
EXECUTABLES, LOOKUP_ERRORS = {}, {}
for name in ("chease", "dcon", "match", "rdcon", "gpec"):
    try:   # a $CHEASEHOME/$GPECHOME pointing at no executable raises; that is "unavailable", not a crash
        EXECUTABLES[name] = find_chease_executable() if name == "chease" else gpec.find_gpec_executable(name)
    except FileNotFoundError as error:
        EXECUTABLES[name], LOOKUP_ERRORS[name] = None, str(error)
NEEDS = {"CHEASE": ("chease",), "DCON": ("dcon", "match"), "RDCON": ("rdcon",), "GPEC": ("dcon", "match", "gpec")}

print(f"mode: {MODE}")
for name, path in EXECUTABLES.items():
    ok, detail = executable_ready(path)
    print(f"  {name:7s} {'ready ' if ok else 'absent'} {LOOKUP_ERRORS.get(name, detail)}")
READY = {}
for stage, programs in NEEDS.items():
    missing = [program for program in programs if not executable_ready(EXECUTABLES[program])[0]]
    READY[stage] = MODE == "lab" and not missing
    if MODE != "lab":
        STAGES[stage].reason = "offline mode"
    elif missing:
        STAGES[stage].execution = "unavailable"
        STAGES[stage].reason = f"missing or not launchable: {', '.join(missing)}"

MANIFEST = mhd_stability_manifest()
LAB_ROOT = Path(tempfile.mkdtemp(prefix="vaft-t05-"))   # short: the GPEC suite reads 128-character paths
# The solver settings of every live run, the same as the samples were built with.
DCON_OPTIONS = gpec.DCONOptions(termbycross_flag=False, con_flag=False)
print(f"\nsamples: {len(MANIFEST['products'])} products, built {MANIFEST['built']} "
      f"(GPEC {MANIFEST['inputs']['gpec']['commit'][:8]}, status {MANIFEST['status']})")
print(format_stage_table(STAGES.values()))

### The equilibrium

One reconstruction feeds everything: the EFIT equilibrium of the 39915 reference sample — the one
sessions 03 and 04 read — at the time the stability samples were built from (read from their
manifest, never typed). Its slice is written once as a g-file, because that is what CHEASE reads;
CHEASE's refinement of it then feeds every solver. The legacy 2023 g-file of the same instant is
kept only for the comparison in Step 1.

In [ ]:
SHOT, TIME_MS = int(MANIFEST["shot"]), int(MANIFEST["time_ms"])   # the slice the samples were built from

reference = vaft.omas.sample_ods(SHOT)   # sessions 03-04: the regenerated EFIT reconstruction
reference_times = np.asarray(reference["equilibrium.time"], dtype=float)
i_same = int(np.argmin(np.abs(reference_times - TIME_MS * 1e-3)))
assert abs(reference_times[i_same] * 1e3 - TIME_MS) < 0.5, "the sample has no slice at the samples' time"

equilibrium = as_equilibrium(reference, time_index=i_same)
eq_geqdsk = from_omas(reference, i_same)   # the same slice as a g-file: what CHEASE reads in lab mode
eq_ods = eq_geqdsk.to_omas()
GEQDSK = LAB_ROOT / f"g0{SHOT}.{TIME_MS:05d}"
write_geqdsk(eq_geqdsk, GEQDSK)

psi_n, q = equilibrium_safety_factor(equilibrium)
b0 = abs(float(eq_ods["equilibrium.vacuum_toroidal_field.b0"][0]))
r0 = float(eq_ods["equilibrium.vacuum_toroidal_field.r0"])
lcfs_r = np.asarray(eq_ods["equilibrium.time_slice.0.boundary.outline.r"], dtype=float)
lcfs_z = np.asarray(eq_ods["equilibrium.time_slice.0.boundary.outline.z"], dtype=float)
if np.hypot(lcfs_r[0] - lcfs_r[-1], lcfs_z[0] - lcfs_z[-1]) < 1e-9:
    lcfs_r, lcfs_z = lcfs_r[:-1], lcfs_z[:-1]   # a closed outline repeats its first point

print(f"shot {SHOT} at {TIME_MS} ms, EFIT slice {i_same} of the sample ODS")
print(f"q on axis {q[0]:.2f}, q at the boundary {q[-1]:.2f}")
print(f"B0 = {b0:.3f} T at R0 = {r0:.3f} m; LCFS R {lcfs_r.min():.3f}-{lcfs_r.max():.3f} m, "
      f"Z {lcfs_z.min():+.3f}-{lcfs_z.max():+.3f} m")

### The 3-D coils

The packaged VEST non-axisymmetric coil geometry: three sets of six sectors, each a single geometric
filament standing for a 20-turn winding. It is mapped into an ODS once, and every excitation in this
session is overlaid on that geometry.

In [ ]:
coil_config = load_vest_3d_coil_config()
coil_ods = ODS()
coils_non_axisymmetric(coil_ods)

print(f"{'set':4s} {'sectors':>7s} {'turns':>5s}  sector angles [deg]      description")
for name, coil_set in coil_config.coil_sets.items():
    angles = ", ".join(f"{a:g}" for a in coil_set.sector_angles_deg)
    print(f"{name:4s} {len(coil_set.filaments):7d} {coil_set.turns:5.0f}  {angles:24s} {coil_set.description}")

## Guided Analysis

Each solver step follows the same pattern: the physical question and its equations, the workflow
diagram, the inputs and assumptions, run-or-load, the IMAS data, registered plots, and what the
result does and does not say.

### Step 1 — Where a perturbation can resonate

A helical perturbation $e^{i(m\theta - n\phi)}$ has the pitch of the field lines where
$q(\psi_s) = m/n$. There, field-line bending costs nothing: it is the place ideal MHD cannot hold
and resistive MHD can tear.

In [ ]:
vaft.diagram.sfl_coordinate_validity()

In [ ]:
vaft.diagram.rational_surface()

In [ ]:
surfaces = {n: find_rational_surfaces(psi_n, q, n) for n in (1, 2, 3)}

figure, axes = plt.subplots(figsize=(8, 4.5))
axes.plot(psi_n, q, color="k", label="q from the EFIT reconstruction")
for (n, found), marker in zip(surfaces.items(), ("o", "s", "^")):
    axes.plot(found["psi_n_rational"], found["q_rational"], marker, linestyle="none",
              label=f"n={n}: {found['m'].size} rational surfaces")
axes.set_xlabel(r"normalized poloidal flux $\psi_N$")
axes.set_ylabel("q")
axes.set_title(f"#{SHOT} at {TIME_MS} ms: where q = m/n")
axes.grid(alpha=0.4)
axes.legend()
plt.show()

n1 = surfaces[1]
print("n=1 rational surfaces:")
for m, where in zip(n1["m"], n1["psi_n_rational"]):
    print(f"  m={m:2d}  psi_N = {where:.3f}")
print(f"gap between the last five n=1 surfaces: {np.round(np.diff(n1['psi_n_rational'])[-4:], 3)}")

In [ ]:
legacy_path = data_path(f"efit/g0{SHOT}.{TIME_MS:05d}")
legacy_psi, legacy_q = equilibrium_safety_factor(as_equilibrium(legacy_path))
legacy_surfaces = find_rational_surfaces(legacy_psi, legacy_q, 1)

figure, axes = plt.subplots(figsize=(8, 4.2))
axes.plot(psi_n, q, "k", label=f"sample ODS (regenerated EFIT): {n1['m'].size} n=1 surfaces")
axes.plot(legacy_psi, legacy_q, "C1--", label=f"legacy 2023 g-file {legacy_path.name}: "
                                                f"{legacy_surfaces['m'].size} n=1 surfaces")
axes.set_xlabel(r"$\psi_N$")
axes.set_ylabel("q")
axes.set_title(f"Two EFIT reconstructions of #{SHOT} at the same time")
axes.grid(alpha=0.4)
axes.legend()
plt.show()
for m in (3, 4, 5):
    here = n1["psi_n_rational"][n1["m"] == m]
    there = legacy_surfaces["psi_n_rational"][legacy_surfaces["m"] == m]
    print(f"q = {m}: psi_N {here[0]:.3f} in the sample ODS, {there[0]:.3f} in the g-file"
          if here.size and there.size else f"q = {m}: missing from one reconstruction")

$q$ climbs from about 2.3 on axis to about 8.4 at the boundary, so a single $n = 1$ has six rational
surfaces, $m = 3$ to 8, and they crowd towards the edge: the last few are a few percent of $\psi_N$
apart. Two consequences follow for everything below:

- there is no $q = 2$ surface — but $q_0$ is the least constrained number of a magnetics-only
  reconstruction (session 03), and an answer that hinges on the core inherits that uncertainty;
- closely spaced edge surfaces mean that islands there need only be small to **overlap** (Step 9).

The two reconstructions agree on the interior $q = 3, 4, 5$ surfaces to within a few percent of
$\psi_N$ but not elsewhere: the legacy file reaches $q \approx 13$ at a larger boundary and has a
$q = 2$ surface near the axis, the regenerated one stops at $q \approx 8.4$. They also disagree on the
plasma itself — the regenerated, magnetics-only reconstruction carries almost no pressure
($p_{\max} \approx 30$ Pa against about 480 Pa). Every number below belongs to *this* reconstruction.

### Step 2 — CHEASE: from a reconstruction to a stability-ready equilibrium

**The question.** EFIT (session 03) solves the *free-boundary* problem: it finds the plasma current
and boundary that best fit the magnetics, on a coarse $(R, Z)$ grid, with $p'$ and $FF'$ in a few
basis functions. A stability code asks for something else — closed nested surfaces up to a
prescribed boundary, a smooth $\psi$ whose second derivatives (current density, shear) are
trustworthy, and a $q$ profile it can integrate to the edge. CHEASE solves the **fixed-boundary**
Grad–Shafranov equation

$$
\Delta^*\psi \equiv R^2\nabla\cdot\frac{\nabla\psi}{R^2} = -\mu_0 R^2 p'(\psi) - F F'(\psi)
$$

inside a boundary *given to it*, with EFIT's $p'$ and $FF'$, on a fine mesh in flux coordinates.

**Inputs and assumptions.**

- *Boundary.* An X-point or separatrix has $q \to \infty$, where SFL coordinates fail, so the
  imposed boundary is the closed EFIT flux surface at $\psi_N = 0.993$ (`CHEASEConfig.target_psin`),
  not the LCFS. The last 0.7 % of the flux is truncated — deliberately and on the record.
- *Profiles.* $p'$ and $FF'$ come from EFIT; CHEASE holds $q$ at $\psi_N = 0.95$ (`ncscal=1`), so
  the total current may move a few percent.
- *Synthetic unstable variant.* The same boundary and profile shapes with the plasma current raised
  to **2.8 × EFIT's** (221 kA), **re-solved by CHEASE** with a total-current constraint
  (`EquilibriumVariation(plasma_current_scale=2.8)`, `ncscal=2`). It is a converged, force-balanced
  equilibrium, not a rescaled flux map: $q$ drops everywhere, $q_{95} \approx 2.3$ and
  $q_{\mathrm{edge}} \approx 2.9$ — just below 3. It exists to give DCON an equilibrium whose answer is
  different (Step 3); it is not a VEST discharge.

In [ ]:
vaft.diagram.chease_coupling()

In [ ]:
efit_ods = eq_ods
chease_ods, chease_geqdsk = {}, {}
CHEASE_CASES = {"stable": 1.0, "high_ip": 2.8}   # plasma-current multiple of the EFIT slice

if READY["CHEASE"]:
    started = time.monotonic()
    try:
        cases = scan_chease(
            GEQDSK,
            [EquilibriumVariation(label=label, plasma_current_scale=scale) for label, scale in CHEASE_CASES.items()],
            config=CHEASEConfig(executable=str(EXECUTABLES["chease"]), create_plot=False),
            workdir=LAB_ROOT / "chease",
        )
        STAGES["CHEASE"].execution = "computed" if all(c.converged for c in cases) else "failed"
        if STAGES["CHEASE"].execution == "computed":
            for case in cases:
                chease_geqdsk[case.variation.label] = Path(case.result.refined_geqdsk)
                chease_ods[case.variation.label] = read_geqdsk(case.result.refined_geqdsk).to_omas()
            STAGES["CHEASE"].source, STAGES["CHEASE"].validation = "live", "passed"
        else:
            STAGES["CHEASE"].reason = "; ".join(f"{c.variation.label}: {c.error}" for c in cases if not c.converged)
    except Exception as error:   # a failed live run is reported, never hidden
        STAGES["CHEASE"].execution, STAGES["CHEASE"].reason = "failed", repr(error)
    print(f"CHEASE {STAGES['CHEASE'].execution} in {time.monotonic() - started:.0f} s")

if not chease_ods:
    chease_ods = {label: mhd_stability_sample(f"chease_{label}") for label in CHEASE_CASES}
    STAGES["CHEASE"].source = "precomputed sample"
    STAGES["CHEASE"].validation = MANIFEST["products"]["chease_stable"]["validation"]

for label, ods in chease_ods.items():
    q_profile = np.asarray(ods["equilibrium.time_slice.0.profiles_1d.q"], dtype=float)
    ip = float(ods["equilibrium.time_slice.0.global_quantities.ip"])
    print(f"{label:12s} q0 = {q_profile[0]:.2f}, q_edge = {q_profile[-1]:.2f}, Ip = {ip / 1e3:.1f} kA")
print(f"EFIT         q0 = {q[0]:.2f}, q_edge = {q[-1]:.2f}, "
      f"Ip = {float(efit_ods['equilibrium.time_slice.0.global_quantities.ip']) / 1e3:.1f} kA")

In [ ]:
SOURCES = [efit_ods, chease_ods["stable"], chease_ods["high_ip"]]
LABELS = ["EFIT (free boundary)", "CHEASE (fixed boundary)", "CHEASE, Ip x 2.8 (synthetic)"]

figure, axes = plt.subplots(1, 3, figsize=(15, 5.2))
vaft.omas.plot_equilibrium_field_psi(efit_ods, ax=axes[0])
axes[0].set_title("EFIT flux map")
vaft.omas.plot_equilibrium_field_psi(chease_ods["stable"], ax=axes[1])
axes[1].set_title("CHEASE flux map (imposed boundary)")
vaft.omas.plot_equilibrium_geometry_boundary(SOURCES[:2], label=LABELS[:2], ax=axes[2])
axes[2].set_title("EFIT LCFS and the boundary CHEASE solved in")
figure.tight_layout()
plt.show()

In [ ]:
figure, axes = plt.subplots(2, 2, figsize=(12, 7.5))
for axis, name in zip(axes.ravel(), ("equilibrium_profile_q", "equilibrium_profile_pressure",
                                     "equilibrium_profile_pprime", "equilibrium_profile_ffprime")):
    vaft.omas.render_plot(name, SOURCES, label=LABELS, coordinate="psi_norm", ax=axis)
figure.suptitle(f"#{SHOT} at {TIME_MS} ms: EFIT and its CHEASE refinements")
figure.tight_layout()
plt.show()

**The Grad–Shafranov residual.** How well does each stored equilibrium satisfy its *own* force
balance? `vaft.omas.compute_grad_shafranov_residual` evaluates $\Delta^*\psi$ by finite differences
on each equilibrium's own $(R, Z)$ grid and compares it with $-\mu_0R^2p' - FF'$ from its own
profiles, over the **same** physical domain ($\psi_N \le 0.99$), with the **same** normalisation
(the RMS source over that domain) and the same 32 flux surfaces.

In [ ]:
residuals = {label: vaft.omas.compute_grad_shafranov_residual(ods, psi_norm_max=0.99)
             for label, ods in zip(LABELS[:2], SOURCES[:2])}

figure, axes = plt.subplots(1, 3, figsize=(15, 4.6), gridspec_kw={"width_ratios": [1.2, 1, 1]})
for label, result in residuals.items():
    axes[0].plot(result.psi_norm, result.relative, "o-", label=label)
    print(f"{label:24s} median |residual| / RMS source = {np.nanmedian(np.abs(result.relative)):.2e}, "
          f"max = {np.nanmax(np.abs(result.relative)):.2e}  ({int(result.mask.sum())} grid points)")
axes[0].set_xlabel(r"$\psi_N$")
axes[0].set_ylabel("RMS residual / RMS source")
axes[0].set_yscale("log")
axes[0].set_title("Force-balance residual per surface")
axes[0].legend()
for axis, (label, result), ods in zip(axes[1:], residuals.items(), SOURCES[:2]):
    grid = ods["equilibrium.time_slice.0.profiles_2d.0.grid"]
    local = np.where(result.mask, (result.delta_star - result.source) / result.scale, np.nan)
    image = axis.pcolormesh(np.asarray(grid["dim1"]), np.asarray(grid["dim2"]), local.T,
                            cmap="RdBu_r", vmin=-0.1, vmax=0.1, shading="auto")
    axis.set_xlim(lcfs_r.min() - 0.02, lcfs_r.max() + 0.02)
    axis.set_ylim(lcfs_z.min() - 0.02, lcfs_z.max() + 0.02)
    axis.set_aspect("equal")
    axis.set_title(f"{label.split(' (')[0]}: local residual / RMS source")
    axis.set_xlabel("R [m]")
figure.colorbar(image, ax=axes[1:], shrink=0.85)
plt.show()

Read the residual for what it is: a statement about **numerical self-consistency** of each stored
$\psi$ with its stored $p'$, $FF'$ — on each file's own grid, with its own finite-difference error
(EFIT's coarse grid smooths $\Delta^*\psi$; CHEASE's 513 × 513 output resolves it, and the radial
streaks in its map are the interpolation from CHEASE's own polar flux mesh onto $(R, Z)$, not force
imbalance in the solution). It is **not** a ranking of which equilibrium is closer to the
discharge: CHEASE inherits EFIT's profiles and EFIT's boundary, so whatever the magnetics did not
constrain — the core $q$ of Step 1 above all — is exactly as uncertain after refinement as before.

What CHEASE *does* buy is the domain the next three codes need: closed nested surfaces to a known
boundary, a $q$ profile that reaches it, and current-density profiles smooth enough to differentiate.
The comparison table the build recorded (relative change in $q$, $p$, $p'$, $FF'$ and $I_p$):

In [ ]:
if STAGES["CHEASE"].source == "precomputed sample":
    for label in CHEASE_CASES:
        check = MANIFEST["products"][f"chease_{label}"]["check"]
        metrics = check["comparison_to_efit"]
        print(f"{label:12s} beta_N {check['beta_n']:.2f}  q95 {check['q95']:.2f}  "
              f"q rms {metrics['q_rms_rel']:.1%}  p' rms {metrics['pprime_rms_rel']:.1%}  "
              f"FF' rms {metrics['ffprim_rms_rel']:.1%}  Ip {metrics['current_rel_diff']:+.1%}  "
              f"(boundary moved {metrics['boundary_rz_rms'] * 1e3:.1f} mm)")
else:
    print("live CHEASE run: the same metrics are in each case's CHEASEResult.comparison")

### Step 3 — Ideal stability: DCON and $\delta W$

**The question.** Does any displacement $\boldsymbol\xi$ lower the ideal-MHD energy? DCON integrates
the Euler–Lagrange equation of $\delta W$ for one $n$ outward from the axis in Hamada coordinates,
applies the ideal jump condition at each rational surface, closes the problem with the vacuum
outside the plasma (free boundary), and reports three energy spectra — plasma $W_p$, vacuum $W_v$,
total $W_t$. The least-stable $W_t$ decides: **$W_t < 0$ means ideal instability, whatever its
magnitude** (#142). MATCH then reconstructs the eigenfunction $\xi_m(\psi)$ from DCON's
`euler.bin` into `solutions.bin`, and VAFT maps both to `mhd_linear`.

**Inputs and assumptions.** The CHEASE equilibria of Step 2, $n = 1$, `con_flag=f` (the ideal jump
condition at each surface, GPEC's own default) and `termbycross_flag=f` (integrate to the edge rather
than stop at the first sign change of the criterion). A verdict counts only when it holds at **two
radial resolutions**, `mpsi` = 256 and 512, with the same dominant harmonic for an unstable mode.

In [ ]:
vaft.diagram.dcon_rdcon_stability()

In [ ]:
vaft.diagram.kink_mode()

In [ ]:
DCON_RESOLUTIONS = (256, 512)
dcon_ods = {}   # (case, mpsi) -> mhd_linear ODS

if READY["DCON"] and STAGES["CHEASE"].source == "live":
    started = time.monotonic()
    try:
        for case in CHEASE_CASES:
            for mpsi in DCON_RESOLUTIONS:
                workdir = LAB_ROOT / f"dcon-{case}-{mpsi}"
                [record] = gpec.run_gpec_suite_case(
                    gpec.GPECCaseInputs(shot=SHOT, time_ms=TIME_MS, geqdsk=chease_geqdsk[case], workdir=workdir),
                    gpec.GPECSuiteConfig(modules=("dcon",), modes=(1,), mpsi=mpsi, verify_outputs=True,
                                         timeout=3600, dcon=DCON_OPTIONS),
                ).records
                problems = gpec.validate_dcon_result(record.workdir, 1, verify_outputs=True)
                if problems:
                    raise RuntimeError(f"{case} mpsi {mpsi}: {'; '.join(problems)}")
                dcon_ods[case, mpsi], _ = build_mhd_linear_ods(shot=SHOT, time_values=[TIME_MS], workdir=workdir,
                                                               modules=("dcon",), modes=(1,))
        STAGES["DCON"].execution, STAGES["DCON"].source = "computed", "live"
    except Exception as error:
        STAGES["DCON"].execution, STAGES["DCON"].reason = "failed", repr(error)
        dcon_ods = {}
    print(f"DCON {STAGES['DCON'].execution} in {time.monotonic() - started:.0f} s")
elif READY["DCON"]:
    STAGES["DCON"].reason = "needs the live CHEASE equilibrium, and CHEASE was not computed"

if not dcon_ods:
    dcon_ods = {(case, mpsi): mhd_stability_sample(f"dcon_{case}_mpsi{mpsi}")
                for case in CHEASE_CASES for mpsi in DCON_RESOLUTIONS}
    STAGES["DCON"].source = "precomputed sample"

# The verdict, read back from the IMAS data -- the same lines whichever mode filled it.
rows = {}
print(f"{'case':12s} {'mpsi':>5s} {'W_t':>10s} {'W_p':>10s} {'W_v':>8s} {'dominant m':>11s}")
for (case, mpsi), ods in dcon_ods.items():
    [row] = extract_dcon_stability(ods)
    m_dom = int(ods["mhd_linear.time_slice.0.toroidal_mode.0.m_pol_dominant"])
    rows[case, mpsi] = row | {"m_dom": m_dom}
    print(f"{case:12s} {mpsi:5d} {row['W_t'].real:+10.4g} {row['W_p'].real:+10.4g} {row['W_v'].real:+8.3g} {m_dom:11d}")

verdicts = {}
for case in CHEASE_CASES:
    signs = {rows[case, m]["W_t"].real < 0 for m in DCON_RESOLUTIONS}
    dominant = {rows[case, m]["m_dom"] for m in DCON_RESOLUTIONS}
    if len(signs) > 1:
        verdicts[case] = "unresolved: the sign of W_t changes with resolution"
    elif signs == {True}:
        verdicts[case] = ("ideally UNSTABLE, dominant m = %d" % dominant.pop()) if len(dominant) == 1 \
            else "unstable, but the dominant m changes with resolution"
    else:
        verdicts[case] = "ideally STABLE at both resolutions"
    print(f"-> {case}: {verdicts[case]}")
STAGES["DCON"].validation = "passed" if (
    verdicts["stable"].startswith("ideally STABLE") and verdicts["high_ip"].startswith("ideally UNSTABLE")
) else "failed"

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(14, 4.8), sharey=True)
for axis, case in zip(axes, CHEASE_CASES):
    vaft.omas.plot_mhd_linear_profile_displacement(dcon_ods[case, 512], ax=axis)
    axis.set_title(f"{case}: W_t = {rows[case, 512]['W_t'].real:+.3g} (mpsi 512)")
figure.suptitle("DCON n=1 least-stable displacement, per poloidal harmonic")
figure.tight_layout()
plt.show()

In [ ]:
vaft.omas.plot_mhd_linear_overview_eigenfunction(dcon_ods["high_ip", 512], figsize=(13, 4.6))
plt.show()

**Why each displacement has the character it has.**

- *Synthetic unstable case ($I_p \times 2.8$, $q_{\mathrm{edge}} \approx 2.91$).* The least-stable
  eigenvector is an **external kink**: its dominant harmonic is $m = 3$, and $\xi_3$ grows
  monotonically all the way to the plasma boundary. With $q_{\mathrm{edge}}$ just below 3 there is no
  $q = 3$ surface inside the plasma to pin it, so the boundary itself can move with the helical
  pitch of the field — and the current gradient, not the pressure (there is almost none), pays for
  it: $W_p \approx -17$, the vacuum term $W_v \approx +1.8$ is far too small to stabilise it. Raising
  $I_p$ walks $q_{\mathrm{edge}}$ down through the rationals, and DCON finds the classic windows just
  below each $m/n$: in the scan behind this case, $q_{\mathrm{edge}} = 4.07$ and $2.7$–$3.0$ are unstable
  while $3.25$–$3.9$ and $4.3$–$5.4$ are not. The same $m$ dominates at both resolutions.
- *Stable case (the CHEASE-refined 39915).* $W_t \approx +7.5$ at both resolutions. Its
  least-stable eigenvector is not a mode the plasma "wants": it is the gentlest displacement the
  energy principle can find, spread over the coupled harmonics with its largest part at the crowded
  edge ($m = 8$, at both resolutions). The sign is the verdict; the shape of a stable eigenvector is
  read with less weight.

The spikes are the ideal jump conditions at the rational surfaces: $\xi_m$ jumps where $q = m/n$.

**Local criteria.** Global $\delta W$ is not the only ideal limit. The Mercier criterion tests
interchange surface by surface, and high-$n$ ballooning is tested on each field line; the
circular $s$–$\alpha$ model shows the idea — magnetic shear $s$ against the normalized pressure
gradient $\alpha$, with a first-stable, an unstable and a second-stable region.

In [ ]:
vaft.diagram.s_alpha_ballooning()

In [ ]:
for shear, alpha in ((1.0, 0.2), (1.0, 1.0), (0.2, 2.5)):
    stable = stability.s_alpha_ballooning_stable(shear, alpha)
    print(f"s = {shear:.1f}, alpha = {alpha:.1f}: {'stable' if stable else 'ballooning UNSTABLE'}")

The packaged `dcon.in` leaves ballooning off (`DCONOptions(bal_flag=False)`), and DCON writes a
criterion it never evaluated as exactly zero — which is also its marginal value. VAFT records what
was requested so that "marginal everywhere" is never confused with "not computed"; turning it on is
`gpec.GPECSuiteConfig(dcon=gpec.DCONOptions(bal_flag=True))`.

### Step 4 — Tearing: rational layers and $\Delta'$

Ideal MHD holds the topology fixed, so the rational surface is where resistivity matters. The
calculation splits the plasma into ideal outer regions and a thin resistive layer, and matches them.

In [ ]:
vaft.diagram.tearing_layer_matching()

In [ ]:
vaft.diagram.delta_prime()

$\Delta'$ is the jump in the outer solution's logarithmic slope. Two outer solutions with the same
value at the surface, one kinked inward (stable) and one outward:

In [ ]:
for label, inside, outside in (("kinked inward", 2.0, -3.0), ("kinked outward", -1.0, 1.5)):
    jump = stability.delta_prime_from_outer_derivatives(1.0, inside, outside)
    print(f"{label:15s} dpsi/dr = {inside:+.1f} inside, {outside:+.1f} outside  ->  "
          f"Delta' = {jump:+.1f} m^-1  ({'tearing drive' if jump > 0 else 'stable'})")

RDCON solves the outer ideal problem with a Galerkin method between the rational surfaces and
returns the **matching data**: the matrix $\Delta'_{ij}$ over the surfaces of one $n$. VAFT maps its
diagonal — the classical, single-surface tearing index — to `ntms.mode[].deltaw[name="classical"]`,
and keeps the surfaces' $\psi_N$, $q$ and the full complex matrix in `mhd_linear.code.parameters`.
It is run on the stable CHEASE equilibrium at three resolutions, because a $\Delta'$ is only as good
as its convergence.

In [ ]:
RDCON_RESOLUTIONS = (256, 512, 1024)
rdcon_ods = {}

if READY["RDCON"] and STAGES["CHEASE"].source == "live":
    started = time.monotonic()
    try:
        for mpsi in RDCON_RESOLUTIONS:
            workdir = LAB_ROOT / f"rdcon-{mpsi}"
            [record] = gpec.run_gpec_suite_case(
                gpec.GPECCaseInputs(shot=SHOT, time_ms=TIME_MS, geqdsk=chease_geqdsk["stable"], workdir=workdir),
                gpec.GPECSuiteConfig(modules=("rdcon",), modes=(1,), mpsi=mpsi, verify_outputs=True, timeout=3600),
            ).records
            if record.status not in ("completed", "stable"):
                raise RuntimeError(f"mpsi {mpsi}: {record.status} {record.reason}")
            rdcon_ods[mpsi], _ = build_mhd_linear_ods(shot=SHOT, time_values=[TIME_MS], workdir=workdir,
                                                      modules=("rdcon",), modes=(1,))
        STAGES["RDCON"].execution, STAGES["RDCON"].source = "computed", "live"
    except Exception as error:
        STAGES["RDCON"].execution, STAGES["RDCON"].reason = "failed", repr(error)
        rdcon_ods = {}
    print(f"RDCON {STAGES['RDCON'].execution} in {time.monotonic() - started:.0f} s")
elif READY["RDCON"]:
    STAGES["RDCON"].reason = "needs the live CHEASE equilibrium, and CHEASE was not computed"

if not rdcon_ods:
    rdcon_ods = {mpsi: mhd_stability_sample(f"rdcon_stable_mpsi{mpsi}") for mpsi in RDCON_RESOLUTIONS}
    STAGES["RDCON"].source = "precomputed sample"

figure, axes = vaft.omas.plot_ntms_overview_delta_prime(rdcon_ods[512], figsize=(9, 7.5))
axes = np.ravel(axes)
axes[1].set_yscale("symlog", linthresh=1.0)
axes[1].set_ylim(-1e4, 1e4)
axes[1].axvspan(0.2, 0.9, color="0.93", zorder=0)
axes[1].text(0.21, 3e3, "interior band", fontsize=9)
plt.show()

In [ ]:
surfaces_by_mpsi = {}
for mpsi, ods in rdcon_ods.items():
    [row] = extract_rdcon_stability(ods)
    surfaces_by_mpsi[mpsi] = {s["m"]: s for s in row["surfaces"]}

print(f"{'m':>3s} {'psi_N':>7s} " + " ".join(f"{'mpsi ' + str(m):>12s}" for m in RDCON_RESOLUTIONS) + "   reading")
consistent = []
for m_pol in sorted(surfaces_by_mpsi[RDCON_RESOLUTIONS[0]]):
    values = [surfaces_by_mpsi[mpsi][m_pol]["delta_prime"].real for mpsi in RDCON_RESOLUTIONS]
    psi = surfaces_by_mpsi[RDCON_RESOLUTIONS[0]][m_pol]["psi_n"]
    same_sign = len({v > 0 for v in values}) == 1
    if psi < 0.2:
        reading = "near the axis: not interpretable"
    elif not same_sign:
        reading = "sign changes with resolution: unresolved"
    else:
        reading = ("classical tearing drive" if values[0] > 0 else "classically stable") + (
            " (crowded edge)" if psi > 0.9 else "")
        consistent.append(m_pol)
    print(f"{m_pol:3d} {psi:7.3f} " + " ".join(f"{v:+12.3g}" for v in values) + f"   {reading}")
STAGES["RDCON"].validation = "passed"   # the products reloaded intact; the physics verdict is the table above
print(f"\nsign-consistent away from the axis at all three resolutions: m = {consistent}")

This is the result to take away from RDCON on this equilibrium, and it is not a tearing verdict:

- **The interior and middle surfaces ($m = 3$ to 7) are negative at all three resolutions** — classically
  stable — but their *magnitudes* move by up to a factor of six between `mpsi` 256 and 1024 ($m = 3$:
  $-5.8$, $-18.7$, $-37.2$). The sign is a statement; the number is not converged.
- **The last surface, $m = 8$ at $\psi_N \approx 0.99$, changes sign with resolution**: it sits on the
  truncated edge, where the layer is not thin compared with its distance to the boundary, and quoting
  any one run's sign there would be quoting the grid.
- **The crowded edge** ($\psi_N > 0.9$) is where the single-surface picture behind the classical
  index is most strained, whichever sign comes out.

**DCON and RDCON answer different questions, and should be allowed to disagree.** An equilibrium
ideally stable at $n = 1$ can still have $\Delta' > 0$ somewhere: tearing releases the same free
energy through reconnection, which ideal MHD forbids. Conversely, the sign of the *diagonal* is not a
stability verdict for coupled surfaces — that needs the full matrix and an inner-layer model (RMATCH,
#797), which this session does not run. STRIDE solves the same matching problem independently and is
the cross-check for any $\Delta'$ you intend to quote.

### Step 5 — From stability to driven response: the VEST 3-D coils

Steps 3 and 4 asked the homogeneous question, $\mathcal L\boldsymbol\xi = 0$. The rest of the
session asks the driven one, $\mathcal L\boldsymbol\xi = \mathbf f_{\mathrm{ext}}$, with the drive
coming from VEST's non-axisymmetric coils. Session 04 drew the *diagnostic* geometry; this is the
*actuator* geometry.

In [ ]:
vaft.omas.plot_coil_3d_geometry_topview(coil_ods)
plt.show()

In [ ]:
vaft.omas.plot_coil_3d_geometry3d(coil_ods)
plt.show()

In [ ]:
figure, axes = plt.subplots(figsize=(4.8, 6.5))
axes.plot(eq_ods["wall.description_2d.0.limiter.unit.0.outline.r"],
          eq_ods["wall.description_2d.0.limiter.unit.0.outline.z"], color="0.4", label="limiter")
axes.plot(lcfs_r, lcfs_z, "k-", label="LCFS")
for name, coil_set in coil_config.coil_sets.items():
    points = np.concatenate([filament.points_xyz for filament in coil_set.filaments])
    axes.plot(np.hypot(points[:, 0], points[:, 1]), points[:, 2], ".", markersize=2, label=name)
axes.set_aspect("equal")
axes.set_xlabel("R [m]")
axes.set_ylabel("Z [m]")
axes.set_title("3-D coils projected on the poloidal plane")
axes.legend(fontsize="small", loc="upper left", bbox_to_anchor=(1.02, 1.0))
plt.show()

UP and LOW are saddle arrays above and below the midplane — in the top view UP lies exactly over
LOW — and MID is a set of circular 12-inch coils on the outboard midplane, about 4 cm outside the
LCFS and by far the closest to the plasma. All three have six sectors, but UP and LOW sit at
$15^\circ, 75^\circ, \dots$ while MID sits at $0^\circ, 60^\circ, \dots$ — a sector's angle belongs to
its coil set, and a current pattern built on the wrong angles is a different pattern.

### Step 6 — A coil-current pattern is a toroidal harmonic

Give sector $k$ at angle $\phi_k$ the current

$$
I_k = I_0 \cos(n\phi_k + \phi_0),
$$

with $I_0$ the amplitude, $n$ the harmonic and $\phi_0$ its toroidal phase. The session uses
$I_0 = 200$ A per turn, the excitation of the 48226 reference run, on 20 turns.

In [ ]:
AMPLITUDE_A = 200.0   # per turn
N_TOR = 1

excitations = [
    CoilExcitation.from_mode(name, AMPLITUDE_A, N_TOR, 0.0, sector_angles_deg=coil_set.sector_angles_deg)
    for name, coil_set in coil_config.coil_sets.items()
]
apply_coil_excitation(coil_ods, excitations)
for excitation in excitations:
    print(f"{excitation.coil_set:4s} I_k = {np.round(excitation.currents_a, 1)} A")

vaft.omas.plot_coil_3d_profile_current(coil_ods)
plt.show()

In [ ]:
vaft.omas.plot_coil_3d_spectrum_current(coil_ods, modes=range(0, 8))
plt.show()

Six sectors resolve $|n| \le 2$; the title says so for everything above. Asking for $n = 5$ or 7
returns the $n = 1$ amplitude again, because on six points $\cos(5\phi_k)$, $\cos(7\phi_k)$ and
$\cos(\phi_k)$ are the *same numbers*. It is the actuator side of session 04's alias step:

```text
session 04: finite toroidal sensors   -> an observable n spectrum modulo the alias step
session 05: finite toroidal actuators -> a controllable n spectrum modulo the same step
```

The coils cannot make a pure $n = 1$ field. What they make is the next step's question.

### Step 7 — The vacuum field at the plasma boundary

The vacuum field of the excited filaments follows from Biot–Savart, with each sector current
multiplied by its 20 turns. Evaluate it on the LCFS at 48 toroidal angles, take the component
normal to the surface — the one that bends field lines out of it — and decompose it toroidally.

In [ ]:
phi_grid = np.linspace(0.0, 2.0 * np.pi, 48, endpoint=False)
tangent_r, tangent_z = np.gradient(lcfs_r), np.gradient(lcfs_z)
length = np.hypot(tangent_r, tangent_z)
normal_r, normal_z = tangent_z / length, -tangent_r / length
outward = np.sign(np.mean(normal_r * (lcfs_r - lcfs_r.mean()) + normal_z * (lcfs_z - lcfs_z.mean())))
normal_r, normal_z = outward * normal_r, outward * normal_z

R, PHI = np.meshgrid(lcfs_r, phi_grid, indexing="ij")
Z = np.broadcast_to(lcfs_z[:, None], R.shape)
probes = np.column_stack([(R * np.cos(PHI)).ravel(), (R * np.sin(PHI)).ravel(), Z.ravel()])

b_normal = {}
for excitation in excitations:
    coil_set = coil_config.coil_sets[excitation.coil_set]
    field = biot_savart_filaments([f.points_xyz for f in coil_set.filaments],
                                  np.asarray(excitation.currents_a) * coil_set.turns, probes).reshape(*R.shape, 3)
    b_r = field[..., 0] * np.cos(PHI) + field[..., 1] * np.sin(PHI)
    b_normal[excitation.coil_set] = b_r * normal_r[:, None] + field[..., 2] * normal_z[:, None]
print(f"{probes.shape[0]} points on the LCFS ({lcfs_r.size} poloidal x {phi_grid.size} toroidal)")

In [ ]:
modes = np.arange(0, 14)
figure, axes = plt.subplots(figsize=(8, 3.8))
for offset, (name, values) in zip((-0.2, 0.0, 0.2), b_normal.items()):
    strongest = int(np.argmax(np.abs(values).max(axis=1)))
    spectrum = toroidal_mode_decomposition(phi_grid, values[strongest], modes)
    # 2|C_n| is the peak amplitude of a cosine for n != 0; the mean (n = 0) is |C_0|.
    amplitude = np.array([(2.0 if n else 1.0) * abs(spectrum[n]) for n in modes]) * 1e4
    axes.plot(modes + offset, amplitude / amplitude[1], "o", label=f"{name}, at its strongest LCFS point")
    print(f"{name:4s} n=1 {amplitude[1]:6.2f} G   n=5/n=1 {amplitude[5] / amplitude[1]:.2f}   "
          f"n=7/n=1 {amplitude[7] / amplitude[1]:.2f}   n=2,3,4 < {amplitude[2:5].max():.1e} G")
axes.set_yscale("log")
axes.set_ylim(1e-4, 5.0)
axes.set_xticks(modes)
axes.set_xlabel("toroidal mode number n of the field")
axes.set_ylabel(r"$|b_{n}| / |b_{1}|$ normal to the LCFS")
axes.set_title("An n=1 current pattern drives n = 1, 5, 7, 11, 13 ...; the rest is rounding")
axes.legend(fontsize="small", loc="lower right")
plt.show()

The field is exactly as the sectors allow: $n = 1$ and its sidebands $n = 6k \pm 1$, and nothing at
$n = 2, 3, 4$ beyond rounding. How strong the sidebands are depends on distance. A high-$n$ field
decays faster away from the coil, so at the LCFS the UP and LOW sidebands are a tenth of $n = 1$ or
less. MID, 4 cm away, is different: its $n = 5$ and $n = 7$ are **as strong as $n = 1$**. Close to a
coil the field is a local bump, not a harmonic.

GPEC takes one $n$ per run, so an $n = 1$ run is the $n = 1$ part of this field and nothing else.
For MID that leaves out as much field at the boundary as it keeps. The sidebands resonate much
farther out, on $q = m/5$ and $m/7$ surfaces, but they are not zero.

In [ ]:
theta = np.degrees(np.arctan2(lcfs_z, lcfs_r - r0))
order = np.argsort(theta)
harmonic = {
    name: np.array([toroidal_mode_decomposition(phi_grid, row, [N_TOR])[N_TOR] for row in values])
    for name, values in b_normal.items()
}

figure, axes = plt.subplots(figsize=(8, 3.8))
for name, coefficient in harmonic.items():
    amplitude = 2.0 * np.abs(coefficient)
    axes.plot(theta[order], amplitude[order] * 1e4, label=name)
    print(f"{name:4s} max |b_n=1| = {amplitude.max() * 1e4:5.1f} G  ->  delta B / B0 = {amplitude.max() / b0:.2%}")
axes.set_xlabel(r"geometric poloidal angle on the LCFS $\theta$ [deg]")
axes.set_ylabel(r"$|b_{n=1}|$ [G]")
axes.set_title("Where each coil set's n=1 field lands on the plasma boundary")
axes.legend()
plt.show()

Each set lands where it sits: UP near $\theta \approx 90^\circ$, LOW near $-90^\circ$, MID on the
outboard midplane. The printed $\delta B/B_0$ is the check that Section 9's linear picture applies:
a percent-level perturbation is small; one of over ten percent — MID at five times this current —
would not be, and a linear response calculation would no longer be the right tool.

### Step 8 — Toroidal phase, relative phase and linearity

**One coil set.** Shifting $\phi_0$ rotates the whole pattern. An axisymmetric plasma cannot tell
one toroidal angle from another, so the response rotates with it and no amplitude changes:
$b_n(\phi_0) = b_n(0)\,e^{i\phi_0}$. Scanning the phase of a single set on its own is not a scan.

**Two coil sets.** The *relative* phase is physical. For UP and LOW driven at the same $n$,

$$
b(\Delta\phi) = b_{\mathrm{UP}} + b_{\mathrm{LOW}}\,e^{i\Delta\phi},
$$

and because the vacuum field — and GPEC's linear response — is linear in the currents,
$\mathbf y = \mathbf R\,\mathbf I$, two calculations give the whole scan. First check that claim
against a direct Biot–Savart evaluation at one phase.

In [ ]:
check_phase = np.pi / 2
low_set = coil_config.coil_sets["LOW"]
shifted = CoilExcitation.from_mode("LOW", AMPLITUDE_A, N_TOR, np.degrees(check_phase),
                                   sector_angles_deg=low_set.sector_angles_deg)
field = biot_savart_filaments([f.points_xyz for f in low_set.filaments],
                              np.asarray(shifted.currents_a) * low_set.turns, probes).reshape(*R.shape, 3)
direct = (field[..., 0] * np.cos(PHI) + field[..., 1] * np.sin(PHI)) * normal_r[:, None] + field[..., 2] * normal_z[:, None]
direct_harmonic = np.array([toroidal_mode_decomposition(phi_grid, row, [N_TOR])[N_TOR] for row in direct])
predicted = harmonic["LOW"] * np.exp(1j * check_phase)
print(f"LOW at phase {np.degrees(check_phase):.0f} deg: largest difference between prediction and direct "
      f"Biot-Savart = {np.abs(direct_harmonic - predicted).max() / np.abs(predicted).max():.1e} of the peak")

In [ ]:
relative = np.radians(np.arange(0.0, 360.0, 15.0))
top, midplane = int(np.argmax(lcfs_z)), int(np.argmax(lcfs_r))
rms, at_top, at_midplane = [], [], []
for delta in relative:
    combined = harmonic["UP"] + harmonic["LOW"] * np.exp(1j * delta)
    rms.append(np.sqrt(np.mean(np.abs(2.0 * combined) ** 2)))
    at_top.append(2.0 * abs(combined[top]))
    at_midplane.append(2.0 * abs(combined[midplane]))

figure, axes = plt.subplots(figsize=(8, 3.8))
axes.plot(np.degrees(relative), np.array(rms) * 1e4, "o-", label="RMS over the whole LCFS")
axes.plot(np.degrees(relative), np.array(at_top) * 1e4, "s--", label="top of the LCFS (UP's side)")
axes.plot(np.degrees(relative), np.array(at_midplane) * 1e4, "^-", label="outboard midplane (both sets)")
axes.set_xlabel(r"relative phase LOW - UP, $\Delta\phi$ [deg]")
axes.set_ylabel(r"$|b_{n=1}|$ [G]")
axes.set_title("Same n, same currents: the relative phase decides where the fields interfere")
axes.legend(fontsize="small")
plt.show()
for label, values in (("RMS over the LCFS", rms), ("top", at_top), ("outboard midplane", at_midplane)):
    values = np.array(values) * 1e4
    print(f"{label:18s} {values.min():5.2f} - {values.max():5.2f} G  (max at {np.degrees(relative[int(np.argmax(values))]):.0f} deg)")

In [ ]:
uniform = np.linspace(-180.0, 180.0, 256, endpoint=False)
poloidal_m = np.arange(-8, 9)
figure, axes = plt.subplots(figsize=(8, 3.8))
for delta, marker in ((0.0, "o"), (np.pi, "s")):
    combined = harmonic["UP"] + harmonic["LOW"] * np.exp(1j * delta)
    on_uniform = np.interp(uniform, theta[order], combined.real[order], period=360.0) \
        + 1j * np.interp(uniform, theta[order], combined.imag[order], period=360.0)
    spectrum = np.fft.fftshift(np.fft.fft(on_uniform)) / uniform.size
    m_axis = np.fft.fftshift(np.fft.fftfreq(uniform.size, d=1.0 / uniform.size)).astype(int)
    pick = np.isin(m_axis, poloidal_m)
    axes.plot(m_axis[pick], 2e4 * np.abs(spectrum[pick]), marker, label=rf"$\Delta\phi$ = {np.degrees(delta):.0f} deg")
axes.set_xticks(poloidal_m)
axes.set_xlabel("poloidal harmonic in the geometric angle (a proxy, not a flux-coordinate m)")
axes.set_ylabel("amplitude [G]")
axes.set_title("In phase and in antiphase, the same n lands in different poloidal harmonics")
axes.legend()
plt.show()

Each set dominates its own side of the boundary, so the top barely notices the relative phase and
neither does the RMS over the whole LCFS. Where the two fields meet — the outboard midplane — they
add at one relative phase and cancel at the opposite one.

The poloidal spectrum says what that means. The field sits in two lobes, near $\theta = +90^\circ$
(UP) and $-90^\circ$ (LOW), half a turn apart in $\theta$. With the lobes in phase the pattern
nearly repeats after half a poloidal turn, and that favours the even harmonics, $|m| = 2, 4, 6$;
with the lobes in antiphase it changes sign after half a turn, and the odd harmonics $|m| = 3, 5$
take over. (Midplane symmetry alone only decides cosine against sine; it is the placement of the two
lobes that selects even or odd $m$.) The same toroidal $n$, the same currents, a different
relative phase — and because each rational surface answers to its own $m = nq$, **the relative
phase chooses which surfaces are driven most**: $q = 2, 4$ in one case, $q = 3, 5$ in the other.

The spectrum here is taken in the geometric angle on the LCFS and in vacuum. A resonant
harmonic lives in a straight-field-line angle on each rational surface, and the plasma changes the
field it sees — both are what GPEC adds.

**Model space and hardware space are not the same.** VAFT and GPEC accept any six currents per set.
Whether VEST's power supplies can drive them — independent channels, amplitude and polarity limits,
phase control, timing, all three sets at once — is a separate, unverified question. Treat every
pattern in this session as a model excitation.

### Step 9 — The plasma's answer: GPEC

**The question.** Given the vacuum field of Step 7, what field does the stable plasma end up with?
GPEC solves $\mathcal L\boldsymbol\xi = \mathbf f_{\mathrm{ext}}$ on the same equilibrium, one coil
set and one $n$ per run, reusing DCON's stability basis (`euler.bin`, `psi_in.bin`, staged into its
working directory with the coil description `coil.in`). Two runs — UP alone and LOW alone, both at
phase zero, 200 A per turn — are enough, by the linearity of Step 8, for the whole relative-phase
scan.

**A setting that decides whether there is a plasma response to show.** The packaged `dcon.in`
ships `con_flag=t` (continue the integration *through* the singular layers). GPEC then switches its
singular-field output off and writes only the **vacuum** resonant field. With `con_flag=f` — the
ideal jump condition, GPEC's own default — and `termbycross_flag=f`, DCON integrates to the edge and
GPEC writes the **total** resonant field on every surface. The build checked that this is a change
of method, not of answer: DCON's $W_t$ moves by 0.6 % (7.504 → 7.461) and GPEC's vacuum energy is
identical. This resolves the gap #1243 recorded.

What VAFT maps to `mhd_linear`: the total perturbed field $J\,\mathbf b\cdot\nabla\psi$ on
$(\psi_N, m)$, the flux-surface geometry, and per rational surface the vacuum resonant flux GPEC
wrote. The total resonant flux is *derived* from the mapped spectrum
(`vaft.omas.compute_gpec_resonant_table`), and reproduces GPEC's own on every surface here to the
printed precision (it has been off by 2.5 % on a near-axis surface of an older reconstruction, where
the field is a hundred times weaker).

In [ ]:
vaft.diagram.gpec_plasma_response()

In [ ]:
GPEC_SETS = ("UP", "LOW")
gpec_ods = {}

if READY["GPEC"] and STAGES["CHEASE"].source == "live":
    started = time.monotonic()
    try:
        for name in GPEC_SETS:
            excitation = next(e for e in excitations if e.coil_set == name)
            coil = gpec.CoilInputSpec.from_excitation(excitation)
            workdir = LAB_ROOT / f"gpec-{name}"
            result = gpec.run_gpec_suite_case(
                gpec.GPECCaseInputs(shot=SHOT, time_ms=TIME_MS, geqdsk=chease_geqdsk["stable"], workdir=workdir),
                gpec.GPECSuiteConfig(modules=("dcon", "gpec"), modes=(1,), mpsi=256, verify_outputs=True,
                                     timeout=3600, dcon=DCON_OPTIONS,
                                     gpec=gpec.IdealGPECOptions(coil_specs=(coil,))),
            )
            records = {record.module: record for record in result.records}
            if records["gpec"].status != "completed":
                raise RuntimeError(f"{name}: {records['gpec'].status} {records['gpec'].reason}")
            print(f"{name}: coil.in staged at {records['gpec'].workdir / 'coil.in'}")
            gpec_ods[name], _ = build_gpec_ideal_ods(shot=SHOT, time_values=[TIME_MS], workdir=workdir, modes=(1,))
        STAGES["GPEC"].execution, STAGES["GPEC"].source = "computed", "live"
    except Exception as error:
        STAGES["GPEC"].execution, STAGES["GPEC"].reason = "failed", repr(error)
        gpec_ods = {}
    print(f"GPEC {STAGES['GPEC'].execution} in {time.monotonic() - started:.0f} s")
elif READY["GPEC"]:
    STAGES["GPEC"].reason = "needs the live CHEASE equilibrium, and CHEASE was not computed"

if not gpec_ods:
    gpec_ods = {name: mhd_stability_sample(f"gpec_{name}") for name in GPEC_SETS}
    STAGES["GPEC"].source = "precomputed sample"

tables = {}
for name, ods in gpec_ods.items():
    try:
        tables[name] = vaft.omas.compute_gpec_resonant_table(ods)
    except ValueError as error:   # a con_flag=t run: vacuum only, and it says so
        print(f"{name}: no total resonant field -- {error}")
STAGES["GPEC"].validation = "passed" if set(tables) == set(GPEC_SETS) else "failed"

for name, table in tables.items():
    print(f"{name}: {len(table['rows'])} rational surfaces, chi1 = {table['chi1']:.4g}")
    print(f"   {'m':>3s} {'psi_N':>6s} {'|Phi_vac| [G]':>14s} {'|Phi_total| [G]':>16s} {'total/vac':>10s}")
    for row in table["rows"]:
        vac, tot = abs(row["Phi_res_vacuum"]), abs(row["Phi_res"])
        print(f"   {row['m_pol']:3d} {row['psi_n']:6.3f} {vac * 1e4:14.3f} {tot * 1e4:16.3f} {tot / vac:10.2f}")

In [ ]:
vaft.omas.plot_mhd_linear_field_spectrum(gpec_ods["UP"])
plt.show()

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(13, 4.4))
for name, marker, shift in zip(tables, ("o", "s"), (-0.002, 0.002)):
    rows_ = tables[name]["rows"]
    psi = [r["psi_n"] + shift for r in rows_]   # UP and LOW coincide: offset so both show
    axes[0].plot(psi, [abs(r["Phi_res_vacuum"]) * 1e4 for r in rows_], marker + "--", label=f"{name}, vacuum")
    axes[0].plot(psi, [abs(r["Phi_res"]) * 1e4 for r in rows_], marker + "-", label=f"{name}, total (with plasma)")
    axes[1].plot(psi, [abs(r["Phi_res"]) / abs(r["Phi_res_vacuum"]) for r in rows_], marker + "-", label=name)
axes[0].set_ylabel(r"$|\Phi_{res}|$ [G]")
axes[0].set_title("Resonant field on each n=1 surface, 200 A per turn")
axes[1].axhline(1.0, color="0.5", linestyle="--", linewidth=0.8)
axes[1].set_ylabel(r"$|\Phi_{res}^{total}| / |\Phi_{res}^{vacuum}|$")
axes[1].set_title("Below one: the plasma screens the resonant field")
for axis in axes:
    axis.set_xlabel(r"$\psi_N$ of the rational surface")
    axis.legend(fontsize="small")
plt.show()

vaft.omas.plot_mhd_linear_profile_island_width(gpec_ods["UP"])
plt.show()

**Screening.** On every $n = 1$ surface the total resonant field is smaller than the vacuum one —
about 0.36–0.51 of it, least screened at the edge. In ideal MHD the plasma answers a resonant
perturbation with a sheet current on the rational surface that opposes it; what is left is the
total. A field that a vacuum calculation would call strong enough to open overlapping islands is,
with the response, about half as strong. Amplification — a total *larger* than the vacuum — would have
pointed at a weakly stable kink coupling to the coil field; this $W_t \approx +7.5$ equilibrium shows
none.

UP drives each surface about 10 % harder than LOW: the regenerated equilibrium is not up–down
symmetric (its LCFS reaches $Z = +0.38$ m but only $-0.36$ m, and the magnetic axis sits 1 cm above the
midplane), so
the two sets differ in size *and* phase — and the relative-phase scan below combines both.

In [ ]:
INTERIOR = (0.2, 0.9)
if set(GPEC_SETS) <= set(tables):
    up, low = tables["UP"]["rows"], tables["LOW"]["rows"]
    chi1 = tables["UP"]["chi1"]
    psi = np.array([r["psi_n"] for r in up])
    geometry = {key: np.array([r[key] for r in up], dtype=float) for key in ("area", "q", "dq_dpsi_n")}
    m_pol = np.array([r["m_pol"] for r in up])
    window = perturbation.ResonantWindow("interior", *INTERIOR, source="Step 4's interpretable band")
    scan = {"vacuum": ([], []), "total": ([], [])}
    for delta in relative:
        for kind, key in (("vacuum", "Phi_res_vacuum"), ("total", "Phi_res")):
            flux = np.array([a[key] for a in up]) + np.array([b[key] for b in low]) * np.exp(1j * delta)
            width = stability.island_width_from_resonant_flux(flux, geometry["area"], geometry["q"],
                                                              geometry["dq_dpsi_n"], m_pol, chi1)
            scan[kind][0].append(perturbation.rms_resonant_field(psi, flux, window=window))
            scan[kind][1].append(perturbation.chirikov(psi, width).max())
    figure, (left, right) = plt.subplots(1, 2, figsize=(12, 4))
    for kind, style in (("vacuum", "o--"), ("total", "s-")):
        left.plot(np.degrees(relative), np.array(scan[kind][0]) * 1e4, style, label=kind)
        right.plot(np.degrees(relative), scan[kind][1], style, label=kind)
    left.set_ylabel("RMS resonant field, interior band [G]")
    right.axhline(1.0, color="r", linestyle="--", linewidth=0.8, label="islands touch")
    right.set_ylabel("largest Chirikov parameter")
    for axis in (left, right):
        axis.set_xlabel(r"relative phase LOW - UP, $\Delta\phi$ [deg]")
        axis.legend()
    figure.suptitle("Two GPEC runs, a whole phase scan: vacuum against total")
    plt.show()
    for kind in scan:
        values = np.array(scan[kind][0])
        print(f"{kind:6s}: interior resonant field largest at {np.degrees(relative[int(np.argmax(values))]):.0f} deg, "
              f"smallest at {np.degrees(relative[int(np.argmin(values))]):.0f} deg; "
              f"largest Chirikov {max(scan[kind][1]):.2f}")

Two lessons, whichever numbers your run prints:

- **The relative phase moves the resonant drive**, surface by surface — exactly the effect of
  Step 7, now on flux-coordinate rational surfaces.
- **The vacuum picture overstates the islands.** With the plasma response the resonant field and
  the island-overlap parameter both drop; whether VEST's crowded edge goes stochastic is a
  question for the *total* field — the reason GPEC exists. Both are still ideal, linear answers:
  rotation, resistivity and the layer physics that decide real penetration are not in them.
- **The response moves the optimum.** The relative phase that drives the interior surfaces most is
  not the same with and without the plasma: here it moves by about a quarter turn (printed above).
  The response changes the phase of each set's resonant field as well as its size, so a coil
  phasing chosen on the vacuum field would be the wrong one — the clearest case in this session
  for computing the response rather than assuming it.

**From scan to optimization.** Write the response as $\mathbf y = \mathbf R\mathbf I$ with
$\mathbf I$ the coil currents and $\mathbf R$ an equilibrium-dependent response matrix. A phase scan
samples one circle of $\mathbf I$; an optimization asks for
$\mathbf I^* = \arg\min_{\mathbf I} J(\mathbf R\mathbf I)$ — minimize the resonant error field
$\sum_m |b^{\mathrm{res}}_{m,n}|^2$, maximize one surface or the edge, subject to current limits.
`vaft.process.perturbation.dominant_mode` gives the leading direction of a coupling matrix;
the full coil phase-space calculation, with current limits and optimization over all sets, is not in
VAFT yet (Open: #1178).

### Step 10 — Integrated interpretation

| | CHEASE | DCON | RDCON | GPEC |
| --- | --- | --- | --- | --- |
| **input** | EFIT g-file: boundary at $\psi_N = 0.993$, $p'$, $FF'$ | CHEASE equilibrium, $n$ | CHEASE equilibrium, $n$ | DCON basis + `coil.in` (set, currents, phase) |
| **assumption** | fixed boundary, $q_{95}$ held, EFIT profiles | ideal MHD, ideal jump at each surface, vacuum outside | thin resistive layers, classical (diagonal) index | ideal, linear, static ($\omega\to0$), one $n$ |
| **diagnostic** | GS residual, profile and $I_p$ changes | sign of $W_t$ at two resolutions, dominant $m$ | $\Delta'$ per surface at three resolutions | total vs vacuum resonant flux, island widths |
| **limit** | core $q$ and $p$ as uncertain as EFIT's; truncated edge | the unstable case is synthetic; a stable eigenvector is read with less weight | magnitudes not converged, edge sign unresolved | no rotation, no resistive penetration physics |
| **IMAS product** | `equilibrium` | `mhd_linear` (energy, eigenfunction) | `ntms` (+ matrix in `mhd_linear.code.parameters`) | `mhd_linear` (field spectrum, surfaces) + `coils_non_axisymmetric` |

The same split runs through the whole session: an **external numerical execution** (the Fortran
codes, optional), a **common data representation** (IMAS IDS, identical whether computed now or
loaded from the sample), and **reusable analysis and plotting** (`vaft.omas.plot_*`,
`extract_*_stability`, `compute_gpec_resonant_table`) that never touch a solver file. Sessions 03
and 04 sit on either side: session 03 made the equilibrium this session refines, session 04 measured
the perturbations these calculations are meant to explain.

In [ ]:
print(format_stage_table(STAGES.values()))

## Interpretation Checkpoints

Each has a definite answer in what you have computed or read.

1. **DCON reports the least-stable $W_t > 0$ at $n = 1$.** Does that make the equilibrium stable to
   tearing? Which step answers that, and why can the two answers differ?
2. **Every interior $\Delta'$ is negative, but its value doubles between resolutions, and the edge
   one changes sign.** What can you say about tearing at this time, and what can you not?
3. **The dominant harmonic of the $n = 1$ eigenfunction is one $m$.** Is the eigenmode a single
   $(m, n)$ component? What did Section 3 say, and what does the eigenfunction plot show?
4. **An $n = 1$ sector pattern drives $n = 5$ and $n = 7$ fields.** Why, and which coil set keeps
   the most of them? What is the corresponding statement in session 04?
5. **Shifting the phase of the MID set alone changes nothing GPEC reports in amplitude.** Why is
   that not a bug, and what kind of phase scan *is* physical?
6. **Two GPEC runs give a 24-point phase scan.** What assumption allows it, and what quantity
   printed in Step 6 tells you the assumption is reasonable here?
7. **The vacuum Chirikov parameter is larger than the total one.** Which physics makes the
   difference, and what is still missing from both before calling the edge stochastic?
9. **CHEASE's GS residual is larger than EFIT's on this metric.** Is the CHEASE equilibrium
   therefore worse? What does the residual measure, and what does it not?
10. **The synthetic $I_p\times2.8$ case is unstable at $m = 3$.** Why does a $q_{edge}$ just below
   3 matter, what makes the case a valid equilibrium to ask DCON about, and why is it still not a
   statement about VEST?
8. **A pattern with arbitrary currents on all 18 sectors gives the smallest resonant field.** What
   must you check before calling it a VEST operating scenario?

## Integrated Analysis

### Part II, level 1 — two times, two reconstructions

Session 04 followed the rational surfaces through the sample ODS over time. A stability answer
depends on the reconstruction *and* the time, and both move it. The packaged legacy g-files cover
two times of 39915; lay them beside the sample ODS at the same times, then — in lab mode — ask DCON
the $n = 1$ question on each legacy file.

In [ ]:
legacy = sorted(data_path("efit").glob(f"g0{SHOT}.*"))
figure, axes = plt.subplots(figsize=(8, 4.2))
for path, colour in zip(legacy, ("C0", "C3")):
    when = int(path.name.split(".")[-1])
    slice_psi, slice_q = equilibrium_safety_factor(as_equilibrium(path))
    index = int(np.argmin(np.abs(reference_times - when * 1e-3)))
    other_psi, other_q = equilibrium_safety_factor(as_equilibrium(reference, time_index=index))
    axes.plot(slice_psi, slice_q, color=colour, label=f"{path.name}")
    axes.plot(other_psi, other_q, color=colour, linestyle="--",
              label=f"sample ODS at {reference_times[index] * 1e3:.0f} ms")
    found = find_rational_surfaces(slice_psi, slice_q, 1)
    q3 = found["psi_n_rational"][found["m"] == 3]
    print(f"{path.name}: q0 = {slice_q[0]:.2f}, n=1 surfaces m = {found['m'].min()}..{found['m'].max()}, "
          + (f"q = 3 at psi_N = {q3[0]:.3f}" if q3.size else "no q = 3 surface at all"))
axes.set_xlabel(r"$\psi_N$")
axes.set_ylabel("q")
axes.legend(fontsize="small")
axes.grid(alpha=0.4)
plt.show()

Two milliseconds apart, the legacy reconstructions disagree about whether the core has a $q = 3$
surface at all: $q_0$ jumps from about 3.5 to about 2. Nothing in the discharge does that in 2 ms;
the core $q$ of a magnetics-only reconstruction is weakly constrained, and it moves with the fit.
The edge surfaces, pinned by the boundary, barely move. A stability result on the core modes —
the $m = 2$ and $m = 3$ surfaces, the least-stable eigenfunction's dominant harmonic — is only as
firm as that core $q$.

In [ ]:
if READY["DCON"]:
    for path in legacy:
        when = int(path.name.split(".")[-1])
        root = LAB_ROOT / f"time-{when}"
        [record] = gpec.run_gpec_suite_case(
            gpec.GPECCaseInputs(shot=SHOT, time_ms=when, geqdsk=path, workdir=root),
            gpec.GPECSuiteConfig(modules=("dcon",), modes=(1,), dcon=DCON_OPTIONS),
        ).records
        if gpec.validate_dcon_result(record.workdir, 1):
            print(f"{path.name}: DCON output unusable")
            continue
        ods, _ = build_mhd_linear_ods(shot=SHOT, time_values=[when], workdir=root, modules=("dcon",), modes=(1,))
        [row] = extract_dcon_stability(ods)
        print(f"{path.name}: least-stable W_t = {row['W_t'].real:+.4g}, "
              f"dominant m = {int(ods['mhd_linear.time_slice.0.toroidal_mode.0.m_pol_dominant'])}")
else:
    print("lab mode only: DCON n=1 straight on each legacy EFIT g-file (no CHEASE step) -- the least-stable\n"
          "W_t per time. Running DCON on a free-boundary reconstruction directly is a shortcut: its\n"
          "truncation is DCON's psihigh, not a CHEASE boundary.")

### Part II, level 2 — more than one $n$

The same six sectors make $n = 2$ and $n = 3$ patterns too. $n = 2$ is still resolved; $n = 3$ is
the Nyquist harmonic of six sectors, where a cosine pattern alternates in sign and its phase is no
longer free.

In [ ]:
for n in (2, 3):
    pattern_ods = ODS()
    coils_non_axisymmetric(pattern_ods)
    apply_coil_excitation(pattern_ods, [
        CoilExcitation.from_mode(name, AMPLITUDE_A, n, 30.0, sector_angles_deg=coil_set.sector_angles_deg)
        for name, coil_set in coil_config.coil_sets.items()
    ])
    vaft.omas.plot_coil_3d_spectrum_current(pattern_ods, modes=range(0, 7))
    plt.show()
    print(f"n={n} rational surfaces: m = {surfaces[n]['m'].min()}..{surfaces[n]['m'].max()} "
          f"({surfaces[n]['m'].size} surfaces)" if n in surfaces else "")

### Part II, level 3 — back to the measurement

| measured in session 04 | computed in session 05 |
| --- | --- |
| frequency $f$, laboratory frame | nothing here: the calculations are static ($\omega \to 0$) |
| $n$ modulo the alias step | one $n$ per DCON/RDCON/GPEC run; the coils' own alias step |
| candidate rational surfaces for that $n$ | $q = m/n$ surfaces and their positions over time |
| $\delta B$ at the probes, SXR and camera structure | eigenfunction $\xi_{m,n}(\psi)$, $\delta B_{m,n}$, island widths |
| a transient as a sequence of numbers | $\delta W$ and $\Delta'$ on the slice before it |

The chain closes only when the model's quantity is taken to the diagnostic: a synthetic Mirnov
signal from the eigenfunction or from GPEC's field at the probe positions. That comparison, not a
verdict on its own, is what identifies a measured mode.

## Independent Exercise

### A stability and response study of one equilibrium

Take the 39915 equilibrium (offline) or a reconstructed equilibrium of your own discharge (lab mode)
and write a short report:

1. **Validity of the equilibrium**: its time, $q$ on axis and at the edge, how well the core $q$
   is constrained (session 03), and what CHEASE changed (GS residual, $q$, $I_p$).
2. **Rational surfaces** for $n = 1$ and $n = 2$, and their spacing near the edge.
3. **DCON** for $n = 1$ and $n = 2$: the least-stable $W_t$, the number of negative eigenvalues,
   the dominant $m$, and the eigenfunction's harmonic content.
4. **RDCON** $\Delta'$ per surface at two or more resolutions, with the interpretable band you
   choose and why.
5. **Why DCON and RDCON may differ** on your case.
6. **A coil pattern** for a chosen coil set and $n$; its sector currents and current spectrum.
7. **The vacuum field** of that pattern on the LCFS: its toroidal sidebands and $\delta B/B_0$.
8. **A relative-phase scan** between two sets, first in vacuum (offline), then with GPEC (lab).
9. **Which phase maximizes or minimizes** your chosen metric, and whether the answer changes between
   the vacuum and the GPEC calculation.
10. **Hardware feasibility**: state separately whether the excitation is known to be achievable.
11. **Comparison with session 04**: which measured $n$ and which surfaces your results speak to,
    and what synthetic signal would make the comparison quantitative.

In [ ]:
# Lab mode: VAFT_TUTORIAL_MODE=lab with $CHEASEHOME and $GPECHOME. Replace the equilibrium, then rerun from the top.
# GEQDSK = Path("<your g-file>")
#
# A DCON n-scan (cost rises steeply with n; RDCON n=2 needs a patched build once msing >= 25):
# suite_config(("dcon",), 256)  ->  replace modes=(1,) by modes=(1, 2) in suite_config
#
# Ballooning on the DCON run:
# gpec.GPECSuiteConfig(modules=("dcon",), modes=(1,), dcon=gpec.DCONOptions(bal_flag=True, termbycross_flag=False))
#
# Another pattern: the MID set alone at n=2 and 45 degrees
# coil_set = coil_config.coil_sets["MID"]
# pattern = CoilExcitation.from_mode("MID", AMPLITUDE_A, 2, 45.0, sector_angles_deg=coil_set.sector_angles_deg)
# apply_coil_excitation(coil_ods, [pattern])
# vaft.omas.plot_coil_3d_spectrum_current(coil_ods)
#
# The relative phase of MID against UP: add "MID" to GPEC_SETS (lab) and combine the two resonant tables
# flux = Phi_UP + Phi_MID * np.exp(1j * delta)

## Takeaways and Next Steps

- Ideal stability, tearing and driven 3-D response are **one linear operator asked three
  questions**: $\mathcal L\boldsymbol\xi = 0$ with $\delta W$ (DCON), the same outer solution matched
  across resistive layers (RDCON, $\Delta'$), and $\mathcal L\boldsymbol\xi = \mathbf f_{\mathrm{ext}}$
  (GPEC). Marginal stability *is* a static perturbed equilibrium.
- An eigenmode is one $n$ with a spectrum of coupled $m$; a verdict is one number, the structure is
  what meets the measurement.
- CHEASE turns a reconstruction into a stability-ready equilibrium; it does not make it more true.
- A DCON verdict is the sign of $W_t$, held at two resolutions; an unstable mode's structure is the
  explanation, a stable eigenvector's shape is not.
- DCON and RDCON can disagree, and a $\Delta'$ near the axis, at the crowded edge, or that changes
  sign with `mpsi` is a resolution statement before it is a physics one.
- Six sectors make $n$ modulo six, in the currents and in the field — the actuator-side alias step.
- A single set's toroidal phase rotates the answer; the **relative** phase between sets changes it.
  Linearity turns two runs into a whole scan.
- The vacuum field is not the field in the plasma: with `con_flag=f` GPEC gives the total resonant
  field, and on this equilibrium the plasma screens it.
- Every solver result reaches the plots through IMAS, so the offline samples and a live run share
  one analysis path.
- The model's excitation space is not VEST's hardware space.

**Method depth**: `notebooks/linear_ideal_stability_analysis_with_dcon.ipynb`,
`notebooks/linear_resistive_stability_analysis_with_rdcon.ipynb`, and the GPEC reference notebook
`notebooks/perturbed_equilibrium_and_3d_response_with_gpec.ipynb`. The single-concept 3-D harmonic
diagrams (`vaft.diagram.complex_harmonic`, `toroidal_harmonic_phase`,
`harmonic_real_space_projection`) illustrate the Fourier picture this session uses.
Open work: DCON closure across native output, IMAS mapping and post-processing (Open: #940);
Mercier and ballooning criteria in the native output and `mhd_linear` (Open: #527); RDCON IMAS
mapping (Open: #939); coil phase-space optimization (Open: #1178).

**Next**: Session 06 leaves the single discharge. It asks which states the machine actually reaches
across many shots — the operational space — and where the limits of this session sit in it.